In [118]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'openset'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    if not os.path.exists(result_file):
        continue
    df = pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['fold_type'] = df['args'].apply(lambda x: x['fold_type'])
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    if method == 'plm_ood':
        df['method'] = df.apply(lambda x: 'Llama3.1-8B' + '-' + x['args']['Detecor'], axis=1)
        # print()
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)>=10)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx', 'fold_type'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx', 'fold_type'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_6": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_type', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
# df_melted = df_melted[(~df_melted['dataset'].isin(['XTopic', 'X-Topic'])) & (df_melted['fold_type'].isin(['fold']))]
df_melted = df_melted[(~df_melted['dataset'].isin(['XTopic', 'X-Topic']))]
df_melted = df_melted.drop_duplicates(['dataset', 'method', 'fold_type', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

  0%|          | 0/2302 [00:00<?, ?it/s]

100%|██████████| 1305/1305 [00:00<00:00, 105800.07it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,fold_type,metric,value
9778,0.1,0.25,20NG,ADB,0,fold,ACC,53.8000
9779,0.1,0.25,20NG,ADB,0,fold,F1,50.7301
9780,0.1,0.25,20NG,ADB,0,fold,K-F1,48.3731
9781,0.1,0.25,20NG,ADB,0,fold,N-F1,62.5150
9846,0.1,0.50,20NG,ADB,0,fold,ACC,60.3900
...,...,...,...,...,...,...,...,...
43417,1.0,0.25,thucnews,clap,3,imbalance_fold,N-F1,0.0000
43462,1.0,0.25,thucnews,clap,4,imbalance_fold,ACC,27.0900
43463,1.0,0.25,thucnews,clap,4,imbalance_fold,F1,33.5381
43464,1.0,0.25,thucnews,clap,4,imbalance_fold,K-F1,0.0000


In [119]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_type', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                              0.1                             \
known_cls_ratio                           0.25                              
metric                                     ACC       F1     K-F1     N-F1   
dataset  method fold_type      fold_idx                                     
20NG     ADB    fold           0         53.80  50.7301  48.3731  62.5150   
                               1         40.82  40.1574  39.6174  42.8571   
                               2         22.19  25.5297  27.3626  16.3651   
                               3         51.45  56.4627  56.9910  53.8210   
                               4         58.17  49.3235  45.7212  67.3349   
...                                        ...      ...      ...      ...   
thucnews clap   imbalance_fold 0           NaN      NaN      NaN      NaN   
                               1           NaN      NaN      NaN      NaN   
                               2           NaN      NaN      NaN      NaN   
                               3           NaN      NaN      NaN      NaN   
                               4           NaN      NaN      NaN      NaN   

labeled_ratio                                                              \
known_cls_ratio                           0.50                              
metric                                     ACC       F1     K-F1     N-F1   
dataset  method fold_type      fold_idx                                     
20NG     ADB    fold           0         60.39  69.6270  71.6784  49.1130   
                               1         66.04  67.4986  67.7088  65.3968   
                               2         48.42  57.5567  60.1148  31.9749   
                               3         62.88  69.5776  71.0790  54.5631   
                               4         80.50  81.5961  81.6761  80.7968   
...                                        ...      ...      ...      ...   
thucnews clap   imbalance_fold 0           NaN      NaN      NaN      NaN   
                               1           NaN      NaN      NaN      NaN   
                               2           NaN      NaN      NaN      NaN   
                               3           NaN      NaN      NaN      NaN   
                               4           NaN      NaN      NaN      NaN   

labeled_ratio                                            ...      1.0  \
known_cls_ratio                           0.75           ...     0.75   
metric                                     ACC       F1  ...     N-F1   
dataset  method fold_type      fold_idx                  ...            
20NG     ADB    fold           0         75.92  82.8313  ...  70.2454   
                               1         75.99  82.3294  ...  37.7682   
                               2         73.44  80.9509  ...  76.9968   
                               3         94.22  95.7202  ...  91.7722   
                               4         88.90  90.8816  ...  78.1302   
...                                        ...      ...  ...      ...   
thucnews clap   imbalance_fold 0           NaN      NaN  ...      NaN   
                               1           NaN      NaN  ...      NaN   
                               2           NaN      NaN  ...      NaN   
                               3           NaN      NaN  ...      NaN   
                               4           NaN      NaN  ...      NaN   

labeled_ratio                                        0.1                   \
known_cls_ratio                                     0.25             0.50   
metric                                  LLM_OOD_Accuracy LLM_OOD_Accuracy   
dataset  method fold_type      fold_idx                                     
20NG     ADB    fold           0                     NaN              NaN   
                               1                     NaN              NaN   
                               2                     NaN              NaN   
                               3                     

In [120]:
# df = pd.read_csv('results/openset/dyen/results2.csv')
# df['args'] = df['args'].apply(lambda x: json.loads(x))
# def func(x):
#     x['fold_type'] = 'fold'
#     return x
# df['args'] = df['args'].apply(lambda x: func(x))
# df['args'] = df['args'].apply(lambda x: json.dumps(x))
# df.to_csv('results/openset/dyen/results.csv', index=None)

In [ ]:
cells = (
    df_melted[
        [
            "dataset",
            "method",
            "labeled_ratio",
            "known_cls_ratio",
            "fold_type",
            "fold_idx",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "method",
            "labeled_ratio",
            "known_cls_ratio",
            "fold_type",
            "fold_idx",
        ]
    )
    .reset_index(drop=True)
)


progress_by_dm = (
    cells.groupby(["dataset", "method", "fold_type"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),
    )
    .reset_index()
)


fold_coverage = (
    cells.groupby(
        ["dataset", "method", "labeled_ratio", "known_cls_ratio", "fold_type"]
    )["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset", "method", "labeled_ratio", "known_cls_ratio"])
)

In [ ]:
EXPECTED_FOLDS = None
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)

    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = (
        fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    )
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method", "fold_type"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text",
    ).sort_index()


fold_list = (
    cells.groupby(
        ["dataset", "method", "labeled_ratio", "known_cls_ratio", "fold_type"]
    )["fold_idx"]
    .apply(lambda s: sorted(pd.unique(s)))
    .reset_index(name="folds_list")
)
fold_list["folds_list_str"] = fold_list["folds_list"].apply(
    lambda x: "[" + ", ".join(map(str, x)) + "]"
)

pivot_folds_list = fold_list.pivot(
    index=["fold_type", "dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str",
).sort_index()

pivot_folds_list.to_excel(f"results/{task}_progress.xlsx")
pivot_folds_list

labeled_ratio                                                       0.1  \
known_cls_ratio                                                    0.25   
fold_type      dataset  method                                            
fold           20NG     ADB                             [0, 1, 2, 3, 4]   
                        DOC                             [0, 1, 2, 3, 4]   
                        DeepUnk                         [0, 1, 2, 3, 4]   
                        DyEn                            [0, 1, 2, 3, 4]   
                        KnnCon                          [0, 1, 2, 3, 4]   
...                                                                 ...   
imbalance_fold thucnews Llama3.1-8B-OpenMax                         NaN   
                        Llama3.1-8B-TemperatureScaling              NaN   
                        SCL                                         [0]   
                        ab                                          [0]   
                        clap                                        NaN   

labeled_ratio                                                            \
known_cls_ratio                                                    0.50   
fold_type      dataset  method                                            
fold           20NG     ADB                             [0, 1, 2, 3, 4]   
                        DOC                             [0, 1, 2, 3, 4]   
                        DeepUnk                         [0, 1, 2, 3, 4]   
                        DyEn                            [0, 1, 2, 3, 4]   
                        KnnCon                          [0, 1, 2, 3, 4]   
...                                                                 ...   
imbalance_fold thucnews Llama3.1-8B-OpenMax                         NaN   
                        Llama3.1-8B-TemperatureScaling              NaN   
                        SCL                                         NaN   
                        ab                                          NaN   
                        clap                                        NaN   

labeled_ratio                                                            \
known_cls_ratio                                                    0.75   
fold_type      dataset  method                                            
fold           20NG     ADB                             [0, 1, 2, 3, 4]   
                        DOC                             [0, 1, 2, 3, 4]   
                        DeepUnk                         [0, 1, 2, 3, 4]   
                        DyEn                            [0, 1, 2, 3, 4]   
                        KnnCon                          [0, 1, 2, 3, 4]   
...                                                                 ...   
imbalance_fold thucnews Llama3.1-8B-OpenMax                         NaN   
                        Llama3.1-8B-TemperatureScaling              NaN   
                        SCL                                         NaN   
                        ab                                          NaN   
                        clap                                        NaN   

labeled_ratio                                                       0.5  \
known_cls_ratio                                                    0.25   
fold_type      dataset  method                                            
fold           20NG     ADB                             [0, 1, 2, 3, 4]   
                        DOC                             [0, 1, 2, 3, 4]   
                        DeepUnk                         [0, 1, 2, 3, 4]   
                        DyEn                            [0, 1, 2, 3, 4]   
                        KnnCon                          [0, 1, 2, 3, 4]   
...                                                                 ...   
imbalance_fold thucnews Llama3.1-8B-OpenMax                         NaN   
                        Llama3.1-8B-TemperatureScaling              NaN   
                      

In [123]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.8097625830959163

In [124]:
all_num

10530

In [125]:
cur_num

8526.8

In [ ]:
dataset_order = ["banking", "clinc", "hwu", "mcid", "stackoverflow"]
metric_order = ["K-F1", "N-F1"]
method_order = [
    "DOC",
    "DeepUnk",
    "ADB",
    "clap",
    "KnnCon",
    "DyEn",
    "Llama3.1-8B-EnergyBased",
    "Llama3.1-8B-Entropy",
    "Llama3.1-8B-KLMatching",
    "Llama3.1-8B-LogitNorm",
    "Llama3.1-8B-Mahalanobis",
    "Llama3.1-8B-MaxLogit",
    "Llama3.1-8B-MaxSoftmax",
    "Llama3.1-8B-OpenMax",
    "Llama3.1-8B-TemperatureScaling",
]


tmp_df = (
    df_melted.groupby(
        ["labeled_ratio", "known_cls_ratio", "dataset", "method", "metric"]
    )
    .mean()
    .reset_index()
)

tmp_df = tmp_df[
    (tmp_df["labeled_ratio"] == 1.0)
    & (tmp_df["dataset"].isin(dataset_order))
    & (tmp_df["metric"].isin(metric_order))
]

tmp_df = tmp_df[tmp_df["method"].isin(method_order)]


tmp_df["dataset"] = pd.Categorical(
    tmp_df["dataset"], categories=dataset_order, ordered=True
)
tmp_df["metric"] = pd.Categorical(
    tmp_df["metric"], categories=metric_order, ordered=True
)
tmp_df["method"] = pd.Categorical(
    tmp_df["method"], categories=method_order, ordered=True
)


tmp_df_pivot = tmp_df.pivot(
    index=["known_cls_ratio", "method"], columns=["dataset", "metric"], values="value"
)


tmp_df_pivot = tmp_df_pivot.sort_index(
    axis=0, level=["known_cls_ratio", "method"]
).sort_index(axis=1, level=["dataset", "metric"])

tmp_df_pivot

TypeError: agg function failed [how->mean,dtype->object]